docker run -d --name=typesense-server -p 8108:8108 -v typesense-data:/data typesense/typesense:0.25.2 --data-dir /data --api-key=xyz --enable-cors
http://localhost:8108
docker run -d --name=typesense-server_2 -p 8108:8108 -v typesense-data:/data typesense/typesense:29.0.rc17-arm64-lg-page16 --data-dir /data --api-key=xyz --enable-cors


# 1. Update your JSONL file with sort_order field (if needed)
01_Creating_typesense_data.ipynb

# 2. Create the collection using the schema
curl -X POST "http://localhost:8108/collections" \
  -H "X-TYPESENSE-API-KEY: xyz" \
  -H "Content-Type: application/json" \
  -d @kpi_schema.json

# 3. Import the documents
curl -X POST "http://localhost:8108/collections/kpi_index/documents/import?action=create" \
  -H "X-TYPESENSE-API-KEY: xyz" \
  -H "Content-Type: application/json" \
  --data-binary @./typesense-data/data/kpi_synonyms_updated.jsonl

# 4. Verify collection creation
curl -X GET "http://localhost:8108/collections/kpi_index" \
  -H "X-TYPESENSE-API-KEY: xyz" 

# 5. Test a search query
curl -X GET "http://localhost:8108/collections/kpi_index/documents/search?q=revenue&query_by=kpi,synonym" \
  -H "X-TYPESENSE-API-KEY: xyz"

In [2]:
import requests
import json

url = "http://localhost:8108/collections/kpi_index/documents/search"
headers = {
    "X-TYPESENSE-API-KEY": "xyz"
}
params = {
    "q": "per pill",
    "query_by": "kpi,synonym"
}

response = requests.get(url, headers=headers, params=params)
data = response.json()

# Pretty-print the result
print(json.dumps(data, indent=2))



{
  "facet_counts": [],
  "found": 1,
  "hits": [
    {
      "document": {
        "id": "42",
        "kpi": "Per pill price",
        "sort_order": 43,
        "synonym": "per pill"
      },
      "highlight": {
        "kpi": {
          "matched_tokens": [
            "Per",
            "pill"
          ],
          "snippet": "<mark>Per</mark> <mark>pill</mark> price"
        },
        "synonym": {
          "matched_tokens": [
            "per",
            "pill"
          ],
          "snippet": "<mark>per</mark> <mark>pill</mark>"
        }
      },
      "highlights": [
        {
          "field": "synonym",
          "matched_tokens": [
            "per",
            "pill"
          ],
          "snippet": "<mark>per</mark> <mark>pill</mark>"
        },
        {
          "field": "kpi",
          "matched_tokens": [
            "Per",
            "pill"
          ],
          "snippet": "<mark>Per</mark> <mark>pill</mark> price"
        }
      ],
      "text_match": 1

In [3]:
#Total result found
print(f"Total Result foudn:{len(data['hits'])}")
#Printing 7hit resul restul
print(f"Printing KPI for x result:{data['hits'][0]['document']['kpi']}")

Total Result foudn:1
Printing KPI for x result:Per pill price


⚙️ Updated KpiMatcher Class with Typesense Search
python
Copy
Edit


In [4]:
import typesense

class KpiMatcher:
    def __init__(self, typesense_host="localhost", port="8108", api_key="xyz"):
        self.client = typesense.Client({
            "nodes": [{
                "host": typesense_host,
                "port": port,
                "protocol": "http"
            }],
            "api_key": api_key,
            "connection_timeout_seconds": 2
        })
        self.stop_words = {'show', 'me', 'the', 'in', 'for', 'of', 'and', 'with', 'by', 'top', 'terms'}

    def _preprocess_text(self, text: str) -> str:
        """Clean and normalize text"""
        text = text.lower()
        text = re.sub(r'[^\w\s]', ' ', text)
        return ' '.join(word for word in text.split() if word not in self.stop_words)

    def find_kpis(self, user_input: str, top_k: int = 30):
        try:
            processed_text = self._preprocess_text(user user_input
            words = processed_text.split()
            for word in words:

                search_parameters = {
                    'q': word,
                    'query_by': 'synonym',
                    'num_typos': 2,
                    'per_page': top_k
                }

            results = self.client.collections['kpi_index'].documents.search(search_parameters)
            matches = [hit['document']['kpi'] for hit in results['hits']]

            return list(dict.fromkeys(matches))  # Removes duplicates while preserving order

        except Exception as e:
            print(f"Error querying Typesense: {e}")
            return []


In [13]:
import typesense
import re

class KpiMatcher:
    def __init__(self, typesense_host="localhost", port="8108", api_key="xyz"):
        self.client = typesense.Client({
            "nodes": [{
                "host": typesense_host,
                "port": port,
                "protocol": "http"
            }],
            "api_key": api_key,
            "connection_timeout_seconds": 2
        })
        self.stop_words = {'show', 'me', 'the', 'in', 'for', 'of', 'and', 'with', 'by', 'top', 'terms'}

    def _preprocess_text(self, text: str) -> str:
        """Clean and normalize text"""
        text = text.lower()
        text = re.sub(r'[^\w\s]', ' ', text)
        return ' '.join(word for word in text.split() if word not in self.stop_words)

    def find_kpis(self, user_input: str, top_k: int = 10):
        try:
            processed_text = self._preprocess_text(user_input)
            words = processed_text.split()
            
            # Initialize an empty list to store all matches
            all_matches = []
            
            # Search for each word in the processed text
            for word in words:
                search_parameters = {
                    'q': word,
                    'query_by': 'synonym',
                    'num_typos': 2,
                    'per_page': top_k
                }
                
                # Perform the search for each word
                results = self.client.collections['kpi_index'].documents.search(search_parameters)
                matches = [hit['document']['kpi'] for hit in results['hits']]
                all_matches.extend(matches)

            # Remove duplicates while preserving order
            return list(dict.fromkeys(all_matches))

        except Exception as e:
            print(f"Error querying Typesense: {e}")
            return []


In [19]:
matcher = KpiMatcher()

# query = "avg realization"
# query    ="sales"
# query   ="discounted"
# query= "Show the top 10 brands in terms of CAGR."
# query= " What is the market share"
# query= "Show me the revenue growth and sales performance"
query= "In the top 50 molecules on sales in india in FY25, show me top 3 moelcules where groeth decreased by 30 percent"
query= "What are the top 10 customers for lelinomide capsules, in NAG on FY25. Also show tthe top 3product group on slaes for each customer"
# query = "India"
matched_kpis = matcher.find_kpis(query)

print("Matched KPIs:", matched_kpis)


Matched KPIs: ['Therapy Area', 'Customer', 'NAG', 'QoQ', 'YoY', 'Q1', 'Product Group', 'LAUNCH_FY', 'New Product', 'Product Name', 'Brand', 'Channel', 'BU_SKU_ID', 'Secondary sales', 'Primary sales', 'Net sales plus cutoff', 'a', 'Revenue']


In [21]:
len(matched_kpis  )

18